In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
 %python
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "etl_vinkos")  
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsvinkosetl")



In [0]:
%python
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/*.txt"

In [0]:
from pyspark.sql.functions import *


# Validar si existen archivos TXT en RAW

#ruta_raw = f"abfss://{container}@{storageName}.dfs.core.windows.net/"

try:

    archivos = dbutils.fs.ls(ruta)

    archivos_txt = [
        a for a in archivos
        if a.name.endswith(".txt")
    ]

    if len(archivos_txt) == 0:
        print("No existen archivos para procesar.")
        dbutils.notebook.exit("SIN_ARCHIVOS")

except Exception:

    print("No existen archivos para procesar.")
    dbutils.notebook.exit("SIN_ARCHIVOS")

# Lectura de todos los TXT


df_estadistica_contenedor = (
    spark.read
        .option("header", "true")
        .csv(ruta)
        .withColumn(
            "archivo_origen",
            regexp_extract(
                input_file_name(),
                r'([^/]+$)',
                1
            )
        )
        .withColumn(
            "fecha_carga",
            current_timestamp()
        )
)

# Si alguna columna no existe la creamos vacía
if "jk" not in df_estadistica_contenedor.columns:
    df_estadistica_contenedor = (
        df_estadistica_contenedor
        .withColumn("jk", lit(None).cast("string"))
    )

if "jyv" not in df_estadistica_contenedor.columns:
    df_estadistica_contenedor = (
        df_estadistica_contenedor
        .withColumn("jyv", lit(None).cast("string"))
    )

if "fgh" not in df_estadistica_contenedor.columns:
    df_estadistica_contenedor = (
        df_estadistica_contenedor
        .withColumn("fgh", lit(None).cast("string"))
    )

# Unificar jk, jyv y fgh en una sola columna llamada jyv
df_estadistica_contenedor = (
    df_estadistica_contenedor
    .withColumn(
        "jyv",
        coalesce(
            col("jk"),
            col("jyv"),
            col("fgh")
        )
    )
)

# Seleccionar únicamente las columnas del DDL Bronze
df_estad_contenedor = (

    df_estadistica_contenedor.select(

        col("email"),

        col("jyv"),

        col("Badmail").alias("badmail"),

        col("Baja").alias("baja"),

        col("Fecha envio").alias("fecha_envio"),

        col("Fecha open").alias("fecha_open"),

        col("Opens").alias("opens"),

        col("Opens virales").alias("opens_virales"),

        col("Fecha click").alias("fecha_click"),

        col("Clicks").alias("clicks"),

        col("Clicks virales").alias("clicks_virales"),

        col("Links").alias("links"),

        col("IPs").alias("ips"),

        col("Navegadores").alias("navegadores"),

        col("Plataformas").alias("plataformas"),

        col("archivo_origen"),

        col("fecha_carga")

    )

)

display(df_estad_contenedor)

#df_estad_contenedor.printSchema()

email,jyv,badmail,baja,fecha_envio,fecha_open,opens,opens_virales,fecha_click,clicks,clicks_virales,links,ips,navegadores,plataformas,archivo_origen,fecha_carga
angelica-daniel@live.com.mx,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
sexi_barbi991@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
abrilgocar@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
pm_andres10@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
dolphin2006_1@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
chasada_10@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
clonroterdam@yahoo.com.mx,null,HARD,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
adri_gonz671@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
chaparritagallinita3@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z
pruebita.software@hotmail.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:13:48.159083Z


In [0]:
#Se escribe en la bitacora los nuevos valores
df_estad_contenedor.write \
    .mode("overwrite") \
    .saveAsTable(
        f"{catalogo}.{esquema}.estadistica_contenedor"
    )



In [0]:
#Se muestra lo que tiene la bitacora de control

df = spark.table(f"{catalogo}.{esquema}.estadistica_contenedor")

display(df)

email,jyv,badmail,baja,fecha_envio,fecha_open,opens,opens_virales,fecha_click,clicks,clicks_virales,links,ips,navegadores,plataformas,archivo_origen,fecha_carga
irmis_20@yahoo.com,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
cachorrita175@yahoo.com,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
ircaballero@prodigy.net.mx,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
irfeda@yahoo.com,null,HARD,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
alexxa_1203@yahoo.com,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
chapis6952@yahoo.com,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
isobel_unison@yahoo.com,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
lezti_2002@yahoo.com,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
itzelorihuela@yahoo.com,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
itzylafresa@yahoo.com,null,null,null,08/02/2013 18:30,-,0,0,-,0,0,-,-,-,-,report_7.txt,2026-07-08T06:13:49.124902Z
